# ISU GSHTD — Bulk Extraction for all 142 VITO Cities

Extracts daily Tmax and Tmin from the ISU GSHTD dataset on GEE for all 142 VITO UrbClim cities.  
Each city is represented as a **50 km radius buffer** around its geocoded centroid.  
All pixels within the buffer are spatially averaged to produce one value per day.  
Results are saved as **Parquet files** (`data/isu_cache/{city}.parquet`) — resume-safe, skips cities already done.

Full date range extracted: **2003-01-01 to 2020-12-31** (complete GSHTD coverage).

In [1]:
!pip install geopy


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import ee
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import time
import warnings
warnings.filterwarnings('ignore')

try:
    from geopy.geocoders import Nominatim
    from geopy.exc import GeocoderTimedOut
    GEOPY_AVAILABLE = True
except ImportError:
    GEOPY_AVAILABLE = False
    print('geopy not installed — install with: pip install geopy')

ee.Initialize(project='tl-cities')
print('GEE initialised')

CACHE_DIR = Path('../data/isu_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

GEE_START   = '2003-01-01'
GEE_END     = '2020-12-31'
BUFFER_M    = 50_000   # 50 km radius
GEE_SCALE   = 1000     # 1 km native resolution
CHUNK_DAYS  = 90       # 3-month extraction chunks

GEE initialised


## 1. City list

In [ ]:
VITO_CITIES = [
    'Accra', 'Addis_Abeba', 'Alicante', 'Amman', 'Amsterdam', 'Antwerp', 'Athens', 'Auckland',
    'Barcelona', 'Bari', 'Basel', 'Belgrado', 'Berlin', 'Bilbao', 'Birmingham', 'Bogota',
    'Bologna', 'Bordeaux', 'Brasov', 'Bratislava', 'Brussels', 'Bucharest', 'Budapest',
    'Buenos_Aires', 'Cairo', 'Cape_Town', 'Charleroi', 'Chennai', 'Cluj_Napoca', 'Cologne',
    'Copenhagen', 'Curitiba', 'Dakar', 'Debrecen', 'Dhaka', 'Dubai', 'Dublin', 'Dusseldorf',
    'Edinburgh', 'Frankfurt_am_Main', 'Gdansk', 'Geneva', 'Genoa', 'Ghent', 'Glasgow',
    'Goteborg', 'Graz', 'Gyor', 'Hamburg', 'Helsinki', 'Ho_Chi_Minh', 'Hong_Kong', 'Houston',
    'Islamabad', 'Istanbul', 'Jakarta', 'Karachi', 'Klaipeda', 'Kosice', 'Krakow', 'Lagos',
    'Leeds', 'Leipzig', 'Liege', 'Lille', 'Lima', 'Lisbon', 'Ljubljana', 'Lodz', 'London',
    'Los_Angeles', 'Luxembourg', 'Lyon', 'Madrid', 'Malaga', 'Marseille', 'Marrakesh',
    'Medina', 'Melbourne', 'Mexico_City', 'Milan', 'Miskolc', 'Mogadishu', 'Montpellier',
    'Moscow', 'Munich', 'Murcia', 'Nairobi', 'Nanjing', 'Nantes', 'Naples', 'New_York',
    'Newcastle', 'Nice', 'Novi_Sad', 'Oslo', 'Padua', 'Palermo', 'Palma_de_Mallorca',
    'Paris', 'Pecs', 'Phoenix', 'Podgorica', 'Porto', 'Prague', 'Rabat', 'Riga', 'Rome',
    'Rotterdam', 'Salvador', 'Santiago', 'Sarajevo', 'Sevilla', 'Singapore', 'Skopje',
    'Sofia', 'Split', 'Stockholm', 'Strasbourg', 'Sydney', 'Szeged', 'Tallinn', 'Tartu',
    'Teheran', 'Thessaloniki', 'Tokyo', 'Toronto', 'Toulouse', 'Trieste', 'Tschwane',
    'Turin', 'Tirana', 'Utrecht', 'Valencia', 'Varna', 'Vienna', 'Vilnius', 'Warsaw',
    'Wroclaw', 'Zagreb', 'Zurich'
]

# Manual overrides — non-English names or cities Nominatim maps to the wrong country
GEOCODE_OVERRIDES = {
    'Addis_Abeba':       ('Addis Ababa',          9.0300,   38.7400),
    'Athens':            ('Athens, Greece',       37.9755,   23.7348),  # Nominatim → Athens, Georgia USA
    'Belgrado':          ('Belgrade',             44.8150,   20.4619),
    'Buenos_Aires':      ('Buenos Aires',        -34.6037,  -58.3816),
    'Cluj_Napoca':       ('Cluj-Napoca',          46.7712,   23.6236),
    'Frankfurt_am_Main': ('Frankfurt am Main',    50.1109,    8.6821),
    'Ho_Chi_Minh':       ('Ho Chi Minh City',    10.8231,  106.6297),
    'Hong_Kong':         ('Hong Kong',            22.3193,  114.1694),
    'Los_Angeles':       ('Los Angeles',          34.0522, -118.2437),
    'Medina':            ('Medina, Saudi Arabia', 24.5247,   39.5692),  # Nominatim → Medina, Texas
    'Mexico_City':       ('Mexico City',          19.4326,  -99.1332),
    'New_York':          ('New York',             40.7128,  -74.0060),
    'Novi_Sad':          ('Novi Sad',             45.2671,   19.8335),
    'Palma_de_Mallorca': ('Palma de Mallorca',    39.5696,    2.6502),
    'Salvador':          ('Salvador, Brazil',    -12.9714,  -38.5014),  # Nominatim → San Salvador, El Salvador
    'Santiago':          ('Santiago, Chile',     -33.4489,  -70.6693),  # Nominatim → Santiago, Costa Rica
    'Teheran':           ('Tehran',               35.6892,   51.3890),
    'Tschwane':          ('Tshwane',             -25.7479,   28.2293),
}

# Always delete the coords cache so overrides are applied fresh on every run
COORDS_FILE = Path('../data/isu_cache/city_coords.csv')
if COORDS_FILE.exists():
    COORDS_FILE.unlink()

print(f'{len(VITO_CITIES)} cities | {len(GEOCODE_OVERRIDES)} manual overrides')

## 2. Geocode cities

Resolves each city name to a (lat, lon) centroid using Nominatim, with manual overrides for names that are non-English or ambiguous. Results are cached to `data/isu_cache/city_coords.csv`.

In [6]:
COORDS_FILE = CACHE_DIR / 'city_coords.csv'

def geocode_cities(cities, overrides):
    if not GEOPY_AVAILABLE:
        raise RuntimeError('geopy required — pip install geopy')

    geolocator = Nominatim(user_agent='heat-risk-research')
    rows = []

    for city in cities:
        if city in overrides:
            label, lat, lon = overrides[city]
            rows.append({'city': city, 'label': label, 'lat': lat, 'lon': lon, 'source': 'manual'})
            continue

        query = city.replace('_', ' ')
        try:
            time.sleep(1.1)  # Nominatim rate limit: 1 req/sec
            loc = geolocator.geocode(query, timeout=10)
            if loc:
                rows.append({'city': city, 'label': query, 'lat': loc.latitude,
                             'lon': loc.longitude, 'source': 'nominatim'})
                print(f'  {city:30s} -> {loc.latitude:.4f}, {loc.longitude:.4f}')
            else:
                print(f'  {city:30s} -> NOT FOUND')
                rows.append({'city': city, 'label': query, 'lat': None, 'lon': None, 'source': 'failed'})
        except Exception as e:
            print(f'  {city:30s} -> ERROR: {e}')
            rows.append({'city': city, 'label': query, 'lat': None, 'lon': None, 'source': 'error'})

    return pd.DataFrame(rows)


if COORDS_FILE.exists():
    city_coords = pd.read_csv(COORDS_FILE)
    print(f'Loaded {len(city_coords)} cities from cache: {COORDS_FILE}')
else:
    print('Geocoding 142 cities via Nominatim (~3 minutes)...')
    city_coords = geocode_cities(VITO_CITIES, GEOCODE_OVERRIDES)
    city_coords.to_csv(COORDS_FILE, index=False)
    print(f'Saved to {COORDS_FILE}')

failed = city_coords[city_coords['lat'].isna()]
if len(failed):
    print(f'\nFailed to geocode {len(failed)} cities — add to GEOCODE_OVERRIDES:')
    print(failed['city'].tolist())
else:
    print(f'\nAll {len(city_coords)} cities geocoded successfully')

city_coords.head(10)

Geocoding 142 cities via Nominatim (~3 minutes)...
  Accra                          -> 5.5571, -0.2012
  Alicante                       -> 38.3436, -0.4882
  Amman                          -> 31.9516, 35.9240
  Amsterdam                      -> 52.3731, 4.8925
  Antwerp                        -> 51.2211, 4.3997
  Athens                         -> 33.9598, -83.3764
  Auckland                       -> -36.8521, 174.7632
  Barcelona                      -> 41.3826, 2.1771
  Bari                           -> 41.1258, 16.8620
  Basel                          -> 47.5581, 7.5878
  Berlin                         -> 52.5174, 13.3951
  Bilbao                         -> 43.2630, -2.9350
  Birmingham                     -> 52.4797, -1.9027
  Bogota                         -> 4.6534, -74.0836
  Bologna                        -> 44.4938, 11.3426
  Bordeaux                       -> 44.8412, -0.5800
  Brasov                         -> 45.6525, 25.6106
  Bratislava                     -> 48.1359, 17.15

,city,label,lat,lon,source
0,Accra,Accra,5.557110,-0.201238,nominatim
1,Addis_Abeba,Addis Ababa,9.030000,38.740000,manual
2,Alicante,Alicante,38.343637,-0.488171,nominatim
3,Amman,Amman,31.951569,35.923963,nominatim
4,Amsterdam,Amsterdam,52.373080,4.892453,nominatim
5,Antwerp,Antwerp,51.221110,4.399708,nominatim
6,Athens,Athens,33.959768,-83.376398,nominatim
7,Auckland,Auckland,-36.852095,174.763180,nominatim
8,Barcelona,Barcelona,41.382580,2.177073,nominatim
9,Bari,Bari,41.125784,16.862029,nominatim


## 3. GEE collection selector

GSHTD is split into regional collections. Select the right one based on city coordinates.

In [4]:
GSHTD_BASE = 'projects/sat-io/open-datasets/global-daily-air-temp'

def get_gshtd_collection(lat, lon):
    if -60 <= lat <= 35 and -120 <= lon <= -30:
        return f'{GSHTD_BASE}/latin_america'
    elif 15 <= lat <= 79 and -170 <= lon <= -50:
        return f'{GSHTD_BASE}/north_america'
    elif -50 <= lat <= -10 and 110 <= lon <= 180:
        return f'{GSHTD_BASE}/australia'
    elif -40 <= lat <= 40 and -20 <= lon <= 55:
        return f'{GSHTD_BASE}/africa'
    else:
        return f'{GSHTD_BASE}/europe_asia'


# Sanity check on a few cities
tests = [
    ('Salvador',  -12.97, -38.50),
    ('London',     51.51,  -0.13),
    ('Nairobi',    -1.29,  36.82),
    ('Tokyo',      35.69, 139.69),
    ('Sydney',    -33.87, 151.21),
    ('Houston',    29.76, -95.37),
]
for name, lat, lon in tests:
    coll = get_gshtd_collection(lat, lon).split('/')[-1]
    print(f'  {name:15s} ({lat:7.2f}, {lon:8.2f})  ->  {coll}')

  Salvador        ( -12.97,   -38.50)  ->  latin_america
  London          (  51.51,    -0.13)  ->  europe_asia
  Nairobi         (  -1.29,    36.82)  ->  africa
  Tokyo           (  35.69,   139.69)  ->  europe_asia
  Sydney          ( -33.87,   151.21)  ->  australia
  Houston         (  29.76,   -95.37)  ->  latin_america


## 4. Extraction function

In [ ]:
def extract_city(city_name, lat, lon,
                 start=None, end=None,
                 buffer_m=None, scale=None, chunk_days=None):
    """
    Extract spatially-averaged daily tmax and tmin from GEE for one city.
    Uses reduceRegion() mapped server-side — one float per day transferred, not pixel data.
    """
    # Resolve defaults here so the function can be defined before cell-1 globals exist
    start      = start      or GEE_START
    end        = end        or GEE_END
    buffer_m   = buffer_m   or BUFFER_M
    scale      = scale      or GEE_SCALE
    chunk_days = chunk_days or CHUNK_DAYS

    roi       = ee.Geometry.Point([lon, lat]).buffer(buffer_m)
    coll_path = get_gshtd_collection(lat, lon)

    def get_typed_collection(prop_type):
        return (
            ee.ImageCollection(coll_path)
            .filterDate(start, end)
            .filterBounds(roi)
            .filter(ee.Filter.eq('prop_type', prop_type))
            .map(lambda img:
                img.select('b1')
                   .divide(10)
                   .rename(prop_type)
                   .set('date', img.date().format('YYYY-MM-dd')))
        )

    def pull_chunks(collection, label):
        frames = []
        cur = datetime.strptime(start, '%Y-%m-%d')
        fin = datetime.strptime(end,   '%Y-%m-%d')

        while cur <= fin:
            chunk_end = min(cur + timedelta(days=chunk_days - 1), fin)
            s = cur.strftime('%Y-%m-%d')
            e = (chunk_end + timedelta(days=1)).strftime('%Y-%m-%d')

            chunk = collection.filterDate(s, e)

            def reduce_img(img):
                mean_val = img.reduceRegion(
                    reducer=ee.Reducer.mean(),
                    geometry=roi,
                    scale=scale,
                    bestEffort=True,
                    maxPixels=1e8
                )
                return ee.Feature(None, {
                    'date':  img.get('date'),
                    label:   mean_val.get(label)
                })

            try:
                fc   = ee.FeatureCollection(chunk.map(reduce_img))
                data = fc.getInfo()['features']
                if data:
                    rows = [{'date': f['properties']['date'],
                             label: f['properties'].get(label)}
                            for f in data]
                    df = pd.DataFrame(rows)
                    df['date']  = pd.to_datetime(df['date'])
                    df[label]   = pd.to_numeric(df[label], errors='coerce')
                    df = df.dropna(subset=[label]).sort_values('date')
                    frames.append(df)
                    print(f'    {s}: {len(df)} days')
            except Exception as ex:
                print(f'    {s} failed: {ex}')

            cur = chunk_end + timedelta(days=1)

        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=['date', label])

    print(f'  tmax...')
    tmax_df = pull_chunks(get_typed_collection('tmax'), 'tmax')
    print(f'  tmin...')
    tmin_df = pull_chunks(get_typed_collection('tmin'), 'tmin')

    merged = tmax_df.merge(tmin_df, on='date', how='inner').sort_values('date').reset_index(drop=True)
    merged['city'] = city_name
    return merged

## 5. Run extraction — all cities

Skips cities already cached. Saves a Parquet file per city as it goes so it's safe to interrupt and resume.  
Expect ~3–5 minutes per city (18 years × 3-month chunks × 2 variables).  
Full run for 142 cities will take several hours — run overnight or in batches using `CITY_SUBSET` below.

In [ ]:
# Set to None to run all cities, or pass a list to run a subset
# e.g. CITY_SUBSET = ['Salvador', 'London', 'Nairobi']
CITY_SUBSET = None

ready = city_coords[city_coords['lat'].notna()].copy()
if CITY_SUBSET:
    ready = ready[ready['city'].isin(CITY_SUBSET)]

print(f'Cities to process: {len(ready)}')
print(f'Already cached:    {len(list(CACHE_DIR.glob("*.parquet")))} city files')
print(f'Output directory:  {CACHE_DIR.resolve()}')
print()

results_log = []

for i, row in ready.iterrows():
    city  = row['city']
    lat   = row['lat']
    lon   = row['lon']
    out   = CACHE_DIR / f'{city}.parquet'

    if out.exists():
        existing = pd.read_parquet(out)
        print(f'[{i+1:3d}/{len(ready)}] {city:30s} — already cached ({len(existing)} days), skipping')
        results_log.append({'city': city, 'status': 'cached', 'days': len(existing)})
        continue

    print(f'[{i+1:3d}/{len(ready)}] {city:30s} ({lat:.3f}, {lon:.3f}) ...')
    t0 = time.time()

    try:
        df = extract_city(city, lat, lon)

        if len(df) == 0:
            print(f'  -> No data returned')
            results_log.append({'city': city, 'status': 'no_data', 'days': 0})
            continue

        df.to_parquet(out, index=False)
        elapsed = time.time() - t0
        print(f'  -> {len(df)} days saved ({elapsed/60:.1f} min)')
        results_log.append({'city': city, 'status': 'ok', 'days': len(df)})

    except Exception as e:
        print(f'  -> FAILED: {e}')
        results_log.append({'city': city, 'status': 'failed', 'days': 0, 'error': str(e)})

    time.sleep(2)  # brief pause between cities

print('\nDone.')

Cities to process: 141
Already cached:    0 city files
Output directory:  /Users/martynclark/heatInsights-notebooks/data/isu_cache

[  1/141] Accra                          (5.557, -0.201) ...


## 6. Extraction summary

In [ ]:
log = pd.DataFrame(results_log)
print('=== Extraction summary ===')
print(log['status'].value_counts().to_string())
print()

# Scan all parquet files and report coverage
cache_files = sorted(CACHE_DIR.glob('*.parquet'))
rows = []
for f in cache_files:
    df = pd.read_parquet(f)
    rows.append({
        'city':     f.stem,
        'days':     len(df),
        'start':    df['date'].min(),
        'end':      df['date'].max(),
        'tmax_mean': df['tmax'].mean().round(2),
        'tmin_mean': df['tmin'].mean().round(2),
        'size_kb':  round(f.stat().st_size / 1024, 1),
    })

coverage = pd.DataFrame(rows)
if len(coverage):
    print(f'{len(coverage)} cities cached, total size: {coverage["size_kb"].sum()/1024:.1f} MB')
    print()
    print(coverage.to_string(index=False))
    coverage.to_csv(CACHE_DIR / 'extraction_summary.csv', index=False)
    print(f'\nSaved summary to {CACHE_DIR}/extraction_summary.csv')

## 7. Quick sanity check — plot one city

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

CHECK_CITY = 'Salvador'

f = CACHE_DIR / f'{CHECK_CITY}.parquet'
if f.exists():
    df = pd.read_parquet(f)
    df['date'] = pd.to_datetime(df['date'])

    fig, ax = plt.subplots(figsize=(16, 4))
    ax.plot(df['date'], df['tmax'], color='#d62728', lw=0.7, alpha=0.8, label='Tmax')
    ax.plot(df['date'], df['tmin'], color='#1f77b4', lw=0.7, alpha=0.8, label='Tmin')
    ax.set_title(f'ISU GSHTD — {CHECK_CITY} ({len(df)} days)', fontweight='bold')
    ax.set_ylabel('Temperature (°C)')
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.legend()
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

    print(f'Tmax: {df.tmax.min():.1f} – {df.tmax.max():.1f} °C  (mean {df.tmax.mean():.1f})')
    print(f'Tmin: {df.tmin.min():.1f} – {df.tmin.max():.1f} °C  (mean {df.tmin.mean():.1f})')
else:
    print(f'{CHECK_CITY} not yet cached — run extraction first')